# Interview Prep — SQL + Python Workspace

This notebook uses **DuckDB** as an in-memory SQL engine and **jupysql** for SQL magic cells.

- `%%sql` — run a full SQL block
- `%sql` — run a single-line SQL query
- Regular Python cells work as normal

---

## Setup — Run this first

In [ ]:
%load_ext sql
%sql duckdb:///:memory:
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

import pandas as pd
import numpy as np
print('Ready!')

---
## Sample Data — Load once, reuse across questions

Modify these tables to match whatever schema the interview question uses.

In [ ]:
%%sql
-- Users table
CREATE OR REPLACE TABLE users (
    user_id     INTEGER,
    name        VARCHAR,
    country     VARCHAR,
    signup_date DATE
);

INSERT INTO users VALUES
    (1, 'Alice',   'US', '2023-01-10'),
    (2, 'Bob',     'CA', '2023-02-15'),
    (3, 'Charlie', 'US', '2023-03-20'),
    (4, 'Diana',   'UK', '2023-04-05'),
    (5, 'Eve',     'US', '2023-04-18');

-- Events table
CREATE OR REPLACE TABLE events (
    event_id   INTEGER,
    user_id    INTEGER,
    event_type VARCHAR,
    event_date DATE
);

INSERT INTO events VALUES
    (1, 1, 'page_view',  '2023-05-01'),
    (2, 1, 'click',      '2023-05-01'),
    (3, 2, 'page_view',  '2023-05-02'),
    (4, 3, 'purchase',   '2023-05-03'),
    (5, 1, 'purchase',   '2023-05-04'),
    (6, 4, 'page_view',  '2023-05-04'),
    (7, 5, 'click',      '2023-05-05'),
    (8, 2, 'purchase',   '2023-05-06'),
    (9, 3, 'page_view',  '2023-05-06'),
    (10,5, 'purchase',   '2023-05-07');

---
## SQL Practice

### Q1 — How many purchases did each country make?

In [ ]:
%%sql
SELECT
    u.country,
    COUNT(*) AS purchase_count
FROM events e
JOIN users u ON e.user_id = u.user_id
WHERE e.event_type = 'purchase'
GROUP BY u.country
ORDER BY purchase_count DESC;

### Q2 — 7-day rolling active users (window function practice)

In [ ]:
%%sql
WITH daily_active AS (
    SELECT
        event_date,
        COUNT(DISTINCT user_id) AS dau
    FROM events
    GROUP BY event_date
)
SELECT
    event_date,
    dau,
    SUM(dau) OVER (
        ORDER BY event_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7d_users
FROM daily_active
ORDER BY event_date;

---
## Python Practice

### Tip: Pull SQL results into pandas for further analysis

In [ ]:
# Query result is already a DataFrame (autopandas = True)
df = %sql SELECT * FROM events
print(df.shape)
df.head()

### Q3 — Two Sum (LeetCode-style)

In [ ]:
def two_sum(nums: list[int], target: int) -> list[int]:
    seen = {}
    for i, n in enumerate(nums):
        complement = target - n
        if complement in seen:
            return [seen[complement], i]
        seen[n] = i
    return []

# Test
assert two_sum([2, 7, 11, 15], 9) == [0, 1]
assert two_sum([3, 2, 4], 6) == [1, 2]
print('All tests passed!')

---
## Product Sense Notes

Use this markdown cell to draft your structured product sense answers.

**Framework:**
1. **Clarify** — What is the goal? Who are the users?
2. **Metric** — What would you measure? (north star, guardrails)
3. **Diagnose** — What could cause a metric drop/spike?
4. **Experiment** — How would you A/B test it?
5. **Tradeoffs** — What are the risks / second-order effects?

---

**Q: DAU dropped 10% overnight. Walk me through your investigation.**

*(Write your answer here)*